# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [1]:
from typing_extensions import TypedDict
class NewsroomState(TypedDict):
    stories: dict   # per-story (Agents 1-4)
    script:  dict   # NEW top-level key (Agent 5 adds this)
    # script = {
    #   "full_text":    str,   # complete script
    #   "word_count":   int,   # verified count
    #   "est_duration": str,   # "74s"
    #   "sections":     dict,  # parsed labels
    #   "stories_used": list,  # [1, 2, 3] selection ranks
    #   "attempt":      int,   # 1 or 2 (audit trail)
    # }

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [2]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [3]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 7
   241.1 vel | climate-gov-was-destroyed-open-data-saved-it
   144.4 vel | telegram-s-t-me-domain-has-been-suspended
   124.9 vel | samsung-will-delete-your-health-data-if-you-don-t-let-them-use-it-to-train-ai
   116.9 vel | apple-s-new-speechanalyzer-api-benchmarked-against-whisper-and-its-predecessor
    82.9 vel | building-and-shipping-mac-and-ios-apps-without-ever-opening-xcode
    64.7 vel | the-real-prices-of-frontier-models-tokens-price-right
    18.8 vel | linux-0-11-rewritten-in-idiomatic-rust-boots-in-qemu


In [4]:
# ── INSPECT after Agent 1 ────────────────────────────────────────────
# Agent 1 fields: title, url, source, category, engagement, velocity
from urllib.parse import urlparse

print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   241.1         371  technology     werd.io                      Climate.gov was destroyed. Open data saved it
   144.4         220  technology     whois.com                    Telegram's t.me domain has been suspended
   124.9         196  technology     neow.in                      Samsung will delete your health data if you d
   116.9         524  technology     get-inscribe.com             Apple's new SpeechAnalyzer API, benchmarked a
    82.9         227  technology     scottwillsey.com             Building and Shipping Mac and iOS Apps Withou
    64.7         167  technology     playcode.io                  The real prices of frontier models. Tokens * 
    18.8          30  technology     github.com                   Linux 0.11 rewritten in idiomatic Rust, boots


---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [3]:
import ollama, subprocess, time

try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(
            ["ollama", "serve"],
            start_new_session=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama started


In [6]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [7]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : climate-gov-was-destroyed-open-data-saved-it
Trafiltura Success ✅  In Loading Content :2167 characters
  [background] gathering snippets for: Climate.gov was destroyed. Open data saved it
  [bg] DDG returned 2 snippets
  [bg] Wikipedia search returns: empty (skipped)
  [background] 2 snippets, synthesizing (content-anchored)...
  [synth] 2 snippets, 491 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 452 chars
Result After Background Syntezing is : In 2020, the Trump Administration drastically cut funding to NOAA, leading to the shutdown of Climate.gov, a vital resource for climate data. The site was taken offline, putting at risk over 15 years' worth of key climate information and resources. However, thanks to the efforts of former NOAA employees Rebecca Lindsey and her sister Anna Eshelman, along with another colleague, the data was preserved and made available on a new website, Climate.us.
  [bg] background: 452 chars
  Agent 2 Ends for: Climate.gov was dest

In [8]:
# Run this in notebook after Agent 2 completes
first = next(iter(call2['stories'].values()))
print("Keys after Agent 2:", list(first.keys()))

Keys after Agent 2: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background']


In [9]:
# ── INSPECT after Agent 2 ────────────────────────────────────────────
# Agent 1 fields + Agent 2 new fields: content, background
from urllib.parse import urlparse

# Agent 1 block (unchanged)
print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

# Agent 2 block (new fields only)
print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   241.1         371  technology     werd.io                      Climate.gov was destroyed. Open data saved it
   144.4         220  technology     whois.com                    Telegram's t.me domain has been suspended
   124.9         196  technology     neow.in                      Samsung will delete your health data if you d
   116.9         524  technology     get-inscribe.com             Apple's new SpeechAnalyzer API, benchmarked a
    82.9         227  technology     scottwillsey.com             Building and Shipping Mac and iOS Apps Withou
    64.7         167  technology     playcode.io                  The real prices of frontier models. Tokens * 
    18.8          30  technology     github.com                   Linux 0.11 rewritten in idiomatic Rust, boots

── Agent 2 new fields ──
  

---
## Agent 3: Fact Checker
Scores each story's credibility on a **-1 to +1 scale**.
Zero is the natural boundary — negative = discard, positive = keep.

| Signal | Base Weight | How |
|--------|-------------|-----|
| `source_score` | 20% | Domain trust map (32 domains, 0.0 to +0.95) |
| `llm_credibility_check` | 60% | gpt-oss-120b → REAL +0.9 / OPINION +0.1 / SPAM -0.7 |
| `cross_verify` | 20% | Exa → DDG → confirms or contradicts via second source |

**Dynamic reweighting** — weights shift when cross-verify fires:
- Contradiction detected → llm 60%→30%, verify 20%→50%
- Confirmation detected  → llm 60%→50%, verify 20%→30%
- Neutral (not found)    → standard weights unchanged

**Key design decisions:**
- `-1 to +1` range — negative range earned by SPAM or credible contradiction
- Threshold = 0.0 — zero is the natural semantic boundary
- SPAM → -0.7 (only genuine negative LLM signal)
- Empty response / Groq failure → 0.0 neutral (never discard on crash)
- Thin content < 500 chars → 0.0 neutral
- Soft discard — story marked `discarded=True`, never deleted (audit trail)
- Cross-verify: Exa (semantic, HN-aware) → DDG fallback → neutral
- Contradiction check uses `compound-mini` (separate quota from 120b)

In [10]:
import time                                    # ← add this line
from agents.agent3 import (
    source_score,
    llm_credibility_check,
    check_credibility,
    cross_verify,
)
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [11]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag    = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        regime  = story.get("_cred_regime", "?")
        vby     = story.get("verified_by", "NONE")
        contra  = "❌ contradicted" if story.get("contradicted") else ""
        print(f"{story['credibility_score']:+.2f} {flag} "
              f"[{regime}] verified={vby} {contra}| {story['title']}")
        time.sleep(1)
    return {"stories": stories}

In [12]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

── CREDIBILITY RESULTS ─────────────────────────────
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'Climate.gov was destroyed. Open data sav' -> 'REAL' -> 0.9
  [verify] neutral -- no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00x0.2 + llm=0.90x0.6 + verify=0.00x0.2 = 0.54
+0.54 ✅ KEEP [neutral] verified=NONE | Climate.gov was destroyed. Open data saved it
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] "Telegram's t.me domain has been suspende" -> 'REAL' -> 0.9
  [verify] neutral -- no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00x0.2 + llm=0.90x0.6 + verify=0.00x0.2 = 0.54
+0.54 ✅ KEEP [neutral] verified=NONE | Telegram's t.me domain has been suspended
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'Samsung will delete your health data if ' -> 'REAL' -> 0.9
  [verify] neutral -- no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00x0.2 + llm=0.90x0.6 + verify=0.00x0.2 = 

In [13]:
# Run this after Agent 3 completes
first = next(iter(call3['stories'].values()))
print("Keys after Agent 3:", list(first.keys()))

Keys after Agent 3: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background', 'credibility_score', 'verified_by', 'contradicted', 'discarded', '_cred_regime', '_weights_used']


In [14]:
for sid, story in call3["stories"].items():
    print(sid)
    print(story)

climate-gov-was-destroyed-open-data-saved-it
{'title': 'Climate.gov was destroyed. Open data saved it', 'url': 'https://werd.io/climate-gov-was-destroyed-open-data-saved-it/', 'source': 'hackernews', 'category': 'technology', 'engagement': 371, 'velocity': 241.1, 'content': 'Climate.gov was destroyed. Open data saved it.\n"After losing their jobs at NOAA, Rebecca Lindsey, her sister and another colleague teamed up to rebuild a pivotal resource the Trump administration took offline."\nLink: Trump dismantled a federal climate website. These women rebuilt it., by Jenae Barnes at The 19th\nThis shouldn’t have been necessary, but is still wonderful to see. Climate.gov had been the go-to resource for climate data, but it went offline when the Trump Administration radically cut NOAA’s funding. At that point:\n“[Rebecca] Lindsey joined forces with former NOAA employees Anna Eshelman, and Mary Lindsey, her older sister, to become the core team behind the deactivated site’s successor, Climate.us

In [15]:
# ── INSPECT after Agent 3 — cumulative (A1 + A2 + A3 fields) ────────
# Keys added: credibility_score, verified_by, contradicted,
#             discarded, _cred_regime, _weights_used
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call3['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")


── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   241.1         371  technology     werd.io                      Climate.gov was destroyed. Open data saved it
   144.4         220  technology     whois.com                    Telegram's t.me domain has been suspended
   124.9         196  technology     neow.in                      Samsung will delete your health data if you d
   116.9         524  technology     get-inscribe.com             Apple's new SpeechAnalyzer API, benchmarked a
    82.9         227  technology     scottwillsey.com             Building and Shipping Mac and iOS Apps Withou
    64.7         167  technology     playcode.io                  The real prices of frontier models. Tokens * 
    18.8          30  technology     github.com                   Linux 0.11 rewritten in idiomatic Rust, boots

── Agent 2 new fields ──
  

In [16]:
from urllib.parse import urlparse
# ── INSPECT: full story details after all 3 agents ──────────────────
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'CONTENT':>8} {'BG':>5} {'REGIME':<14} {'VERIFIED_BY':<19} {'ORIGINAL SOURCE':<25} {'TITLE':<50}")
print("-" * 145)



for sid, story in call3["stories"].items():
    flag    = "🗑️" if story.get("discarded") else "✅"
    score   = story.get("credibility_score", 0)
    vel     = story.get("velocity", 0)
    clen    = len(story.get("content", ""))
    bglen   = len(story.get("background", ""))
    regime  = story.get("_cred_regime", "?")
    vby     = story.get("verified_by", "NONE")
    contra  = "❌" if story.get("contradicted") else ""
    #weights = story.get("_weights_used", "?")
    title   = story.get("title", "")
    source = story.get("url", "")
    source_domain = urlparse(
            source
        ).netloc.replace("www.", "")


    print(    f"{flag}  {score:+.2f}  {vel:>6.1f}  {clen:>6}c  "
            f"{bglen:>4}c  {regime:<14} {contra}{vby:<20}"
            f"{source_domain:<25} {title:<50}")

FLAG SCORE      VEL  CONTENT    BG REGIME         VERIFIED_BY         ORIGINAL SOURCE           TITLE                                             
-------------------------------------------------------------------------------------------------------------------------------------------------
✅  +0.54   241.1    2167c   452c  neutral        NONE                werd.io                   Climate.gov was destroyed. Open data saved it     
✅  +0.54   144.4    6000c   358c  neutral        NONE                whois.com                 Telegram's t.me domain has been suspended         
✅  +0.54   124.9    2336c   601c  neutral        NONE                neow.in                   Samsung will delete your health data if you don't let them use it to train AI
✅  +0.54   116.9    6000c   866c  neutral        NONE                get-inscribe.com          Apple's new SpeechAnalyzer API, benchmarked against Whisper and its predecessor
🗑️  -0.03    82.9    6000c   493c  contradiction  ❌github.com      

### Save story : till agent 3 (quality data progress saved to disk) so even if some changes comes in agent 1 to agent 3 , agent 4+ code developement will not be affect

In [17]:

# Save to disk — skip re-processing next time
from agent_tools.story_cache import save_stories
save_stories(call3["stories"])

# TO track log of particular function after a hit of particular no. of times pop-up will come


  [cache] saved 7 stories → data/stories_cache.json (23 total in cache)


In [18]:
# importing story cache
# Instead of running A1 → A2 → A3 every time:

from agent_tools.story_cache import load_stories

# Load the saved run (read only — never write again)
cached_stories = load_stories()
print(f"Loaded {len(cached_stories)} stories from cache")


  [cache] loaded 23 stories from data/stories_cache.json
Loaded 23 stories from cache


---
## Agent 4: Editorial

Selects the top stories to cover today from Agent 3's credibility-scored pool.

**Responsibilities:**
1. **Filter** — remove Agent 3 discards (credibility_score < 0.0)
2. **Score** — editorial_score = cred×0.50 + vel_norm×0.30 + bg_norm×0.20
3. **Deduplicate** — phi3.5 clusters titles by topic (one call, no bleed)
4. **Select** — top 3 by editorial_score with topic diversity enforced
5. **Route** — ≥1 story → Agent 5 · 0 stories → end + macOS notification

**First conditional edge in the pipeline** — all previous agents were linear.

In [19]:
# feed directly into Agent 4
from agents.agent4 import editorial_node, route_after_editorial
call4 = editorial_node(call3)
route = route_after_editorial(call4)
print(f"\nRoute: {route}")

AGENT 4: Editorial
  [filter] 7 stories in → 6 eligible (1 discarded by Agent 3)

  [editorial] max velocity this batch: 241.1
  [editorial] Climate.gov was destroyed. Open data saved it cred=+0.54 vel=1.00 bg=0.56 → 0.683
  [editorial] Telegram's t.me domain has been suspended     cred=+0.54 vel=0.60 bg=0.45 → 0.539
  [editorial] Samsung will delete your health data if you d cred=+0.54 vel=0.52 bg=0.75 → 0.576
  [editorial] Apple's new SpeechAnalyzer API, benchmarked a cred=+0.54 vel=0.48 bg=1.00 → 0.615
  [editorial] The real prices of frontier models. Tokens *  cred=+0.69 vel=0.27 bg=0.81 → 0.587
  [editorial] Linux 0.11 rewritten in idiomatic Rust, boots cred=+0.73 vel=0.08 bg=1.00 → 0.588

  [deduplicate] qwen2.5:7b raw output: ```json
[[1], [2], [3], [4], [5], [6]]
```
  [deduplicate] 6 topic clusters found
  [deduplicate] cluster 0 best → Climate.gov was destroyed. Open data saved it
  [deduplicate] cluster 1 best → Telegram's t.me domain has been suspended
  [deduplicate] clust

In [20]:
# ── INSPECT after Agent 4 — cumulative (A1 + A2 + A3 + A4 fields) ───
# Keys added: editorial_score, selected, selection_rank,
#             selection_reason, _vel_norm, _bg_norm,
#             _topic_cluster, _is_duplicate
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call4['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call4['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")

print()
print('── Agent 4 new fields ──')
print(f"{'SEL':<5} {'RANK':<5} {'ED_SCORE':<10} {'VEL_N':<7} {'BG_N':<7} "
      f"{'CLUSTER':<9} {'IS_DUPLICATE':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 110)
for sid, story in call4['stories'].items():
    selected  = '✅' if story.get('selected') else '—'
    rank      = story.get('selection_rank', '-')
    ed_score  = story.get('editorial_score', 0)
    vel_n     = story.get('_vel_norm', 0)
    bg_n      = story.get('_bg_norm', 0)
    cluster   = story.get('_topic_cluster', '-')
    is_dup    = '❌' if story.get('_is_duplicate') else '✅'
    domain    = urlparse(story.get('url','')).netloc.replace('www.','')
    title     = story.get('title','')[:40]
    print(f"{selected:<5} #{rank!s:<4} {ed_score:<10.3f} {vel_n:<7.3f} {bg_n:<7.3f} "
          f"{cluster!s:<9} {is_dup:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
   241.1         371  technology     werd.io                      Climate.gov was destroyed. Open data saved it
   144.4         220  technology     whois.com                    Telegram's t.me domain has been suspended
   124.9         196  technology     neow.in                      Samsung will delete your health data if you d
   116.9         524  technology     get-inscribe.com             Apple's new SpeechAnalyzer API, benchmarked a
    82.9         227  technology     scottwillsey.com             Building and Shipping Mac and iOS Apps Withou
    64.7         167  technology     playcode.io                  The real prices of frontier models. Tokens * 
    18.8          30  technology     github.com                   Linux 0.11 rewritten in idiomatic Rust, boots

── Agent 2 new fields ──
  

In [21]:
# ── Save checkpoint after Agent 5 ────────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint, list_checkpoints

save_checkpoint(call4, name="till-agent4")
print(f"Available checkpoints: {list_checkpoints()}")

[checkpoint] saved 'till-agent4' → data/checkpoints/till-agent4.json
[checkpoint]   stories: 7
Available checkpoints: ['till-agent4', 'till-agent5', 'till-agent6', 'till-agent6_1', 'till-agent7', 'till-agent8']


---
## Agent 5: Script Writer
Converts the top 3 selected stories into a 60-90 second YouTube Shorts script.

**Model:** llama-3.3-70b-versatile (Groq) — separate quota from Agent 3
**Format:** HOOK → S1_CONTEXT → S1_CORE → S1_TWIST → S2_HOOK → ... → CTA
**Word target:** 150-225 words (~60-90 seconds at spoken pace)
**Tone calibration:** credibility_score from Agent 3 controls framing confidence

In [22]:
from agents.agent5 import script_writer_node

# Run Agent 5 on Agent 4's output
call5 = script_writer_node(call4)

# Print the full script
print("\n── FULL SCRIPT ──────────────────────────────────────────────────")
print(call5["script"]["full_text"])
print(f"\n── STATS ────────────────────────────────────────────────────────")
print(f"Words:    {call5['script']['word_count']}")
print(f"Duration: ~{call5['script']['est_duration']}")
print(f"Sections: {len(call5['script']['sections'])}")
print(f"Stories:  ranks {call5['script']['stories_used']}")
print(f"Attempts: {call5['script']['attempt']}")

AGENT 5: Script Writer
  [script] 3 selected stories found
  [script]   #1 ed=0.683 | Climate.gov was destroyed. Open data saved it
  [script]   #2 ed=0.615 | Apple's new SpeechAnalyzer API, benchmarked against Whi
  [script]   #3 ed=0.588 | Linux 0.11 rewritten in idiomatic Rust, boots in QEMU
  [script] prompt built (8406 chars, 3 stories)
  [script] LLM response: 1472 chars, ~221 words (before enforcement)
  [script] word count after attempt 1: 221 words
  [script] within target (150-225)
  [script] parsed 10 sections: ['HOOK', 'S1_CONTEXT', 'S1_CORE', 'S1_TWIST', 'S2_HOOK', 'S2_CORE', 'S2_TWIST', 'S3_HOOK', 'S3_CORE', 'CTA']

  [script] script complete
  [script]    words:    221
  [script]    duration: ~88s
  [script]    sections: 10
  [script]    attempts: 1
  [script]    stories:  [1, 2, 3]

── FULL SCRIPT ──────────────────────────────────────────────────
HOOK: Climate.gov was destroyed, but open data saved it with over 15 years of key climate information preserved.
S1_CONTEXT:

---
## 🗂️ Cache & Checkpoint Reference

### `story_cache.py` — A1+A2+A3 output
Reload to test Agent 4 or Agent 5 without re-running A1-A3.
```python
from agent_tools.story_cache import load_stories
stories = load_stories()
call4 = editorial_node({"stories": stories})
```
File: `data/stories_cache.json`

### `pipeline_cache.py` — A1+A2+A3+A4+A5 output
Reload to test Agent 6+ without re-running A1-A5.
```python
from agent_tools.pipeline_cache import load_checkpoint
state = load_checkpoint("till-agent5")
call6 = script_qc_node(state)
```
File: `data/checkpoints/till-agent5.json`

| Working on... | Load this | Contains |
|---|---|---|
| Agent 4 or 5 | `load_stories()` | stories only |
| Agent 6+ | `load_checkpoint("till-agent5")` | stories + script |

---

In [23]:
# ── Save checkpoint after Agent 5 ────────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint, list_checkpoints

save_checkpoint(call5, name="till-agent5")
print(f"Available checkpoints: {list_checkpoints()}")

[checkpoint] saved 'till-agent5' → data/checkpoints/till-agent5.json
[checkpoint]   stories: 7
[checkpoint]   script:  221 words, ~88s
Available checkpoints: ['till-agent4', 'till-agent5', 'till-agent6', 'till-agent6_1', 'till-agent7', 'till-agent8']


---
## Agent 6: Script QC + Agent 6.1: Voice-Over Generator

Two separate agents with a clean division of labor — text correctness
vs. audio generation reliability.

### Agent 6 — Script QC (text intelligence)
Validates and polishes Agent 5's script until it's genuinely ready to
become audio. Two-stage model split, mirrors Agent 3's design:

| Stage | Model | Job |
|---|---|---|
| JUDGE | `gpt-oss-120b` | Finds problems: word count, AI-voice phrasing, weak transitions, restated twists, CTA format |
| REWRITE | `llama-3.3-70b-versatile` | Fixes ONLY flagged sections — never regenerates the whole script |

**Also handles (pure Python, no LLM cost):**
- Date humanization ("August 23, 2024" → "last year") — uses the real
  current date, since Agent 5 has no clock
- Deterministic `annotated_text` / `tts_ready_text` split — pacing
  markers are derived from sentence structure, never generated by an
  LLM, and **never sent to Kokoro** (Kokoro can't render `[PAUSE]`/
  `[EMPHASIS]` tags — see KNOWN_ISSUES ISSUE-13)

**Control:** APPROVE/REVISE loop, max 2 iterations. Never blocks the
pipeline — approves with a logged warning if issues remain after 2 tries.

### Agent 6.1 — Voice-Over Generator (audio reliability)
Converts Agent 6's QC'd `tts_ready_text` into actual voice-over audio
using **Kokoro** (via `mlx-audio`, Apple Metal GPU acceleration, $0 cost,
fully local — see KNOWN_ISSUES ISSUE-12).

Not in the original 10-agent roadmap — added after Agent 1 itself
surfaced the Kokoro HN story, Agent 3 confirmed it, and Agent 4 selected
it for scripting. Kept **separate** from Agent 6 because text QC and
audio generation are genuinely different concerns with different
failure modes — Agent 6.1 could later swap TTS backends without
touching Agent 6 at all.

**Reliability guarantee, not just "did the command exit 0":**
generates audio, then verifies actual duration roughly matches the
word-count estimate (±50%) — catches garbled or truncated speech that
a bare success/failure check would miss. Only runs if Agent 6 approved
the script — trusts Agent 6's QC gate completely.

In [24]:
from agent_tools.pipeline_cache import load_checkpoint
from agents.agent6 import script_qc_node

state = load_checkpoint("till-agent5")
call6 = script_qc_node(state)

print("\n── ANNOTATED TEXT ──────────────────────────────────")
print(call6["script"]["annotated_text"])
print("\n── TTS-READY TEXT ──────────────────────────────────")
print(call6["script"]["tts_ready_text"])
print(f"\nApproved: {call6['script']['approved']}")
print(f"Iterations: {call6['script']['iterations']}")
print(f"QC notes: {call6['script']['qc_notes']}")

[checkpoint] loaded 'till-agent5' (saved: 2026-07-13T21:59:47.751125+00:00)
[checkpoint]   stories: 7
[checkpoint]   script:  221 words, ~88s
AGENT 6: Script QC
  [qc] date humanization applied to 10 sections

  [qc] --- iteration 1 ---
  [qc] JUDGE empty content, finish_reason=length
  [qc] JUDGE empty/failed on gpt-oss-120b -> falling back to local qwen2.5:7b
  [qc] JUDGE raw: FLAGGED_SECTIONS: S1_CONTEXT,S1_CORE,S2_HOOK,S3_HOOK
FLAGGED_REASONS: NONE|S1_CONTEXT|S2_HOOK|S3_HOOK
CTA_CATEGORY: A
CTA_OK: YES
  [qc] word_count_ok=True flagged=['S1_CONTEXT', 'S1_CORE', 'S2_HOOK', 'S3_HOOK'] cta_ok=True (category A)
  [qc] rewrote S1_CONTEXT
  [qc] rewrote S1_CORE
  [qc] rewrote S2_HOOK
  [qc] rewrote S3_HOOK

  [qc] --- iteration 2 ---
  [qc] JUDGE empty content, finish_reason=length
  [qc] JUDGE empty/failed on gpt-oss-120b -> falling back to local qwen2.5:7b
  [qc] JUDGE raw: FLAGGED_SECTIONS: S1_CONTEXT,S1_CORE,S2_HOOK,S3_HOOK
FLAGGED_REASONS: NONE|S1_CONTEXT|S2_HOOK|S3_HOOK
CTA_CATEGOR

In [25]:
from agent_tools.pipeline_cache import save_checkpoint
save_checkpoint(call6,'till-agent6')

[checkpoint] saved 'till-agent6' → data/checkpoints/till-agent6.json
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s


'data/checkpoints/till-agent6.json'

In [26]:

from agents.agent6_1 import voice_over_node
from agent_tools.pipeline_cache import load_checkpoint

pt=load_checkpoint('till-agent6')

call6_1 = voice_over_node(pt)

print(f"Audio generated: {call6_1['script']['audio_generated']}")
print(f"Audio files stitched: {call6_1['script']['audio_chunks']}")
print(f"Duration: {call6_1['script']['audio_duration']}s")
print(f"Verified: {call6_1['script']['duration_verified']}")

[checkpoint] loaded 'till-agent6' (saved: 2026-07-13T22:00:14.450663+00:00)
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s
AGENT 6.1: Voice-Over Generator (Kokoro)
  [voiceover] generating audio (260 words total)...
  [voiceover] split script into 9 TTS-safe text chunk(s) (max 40 words each)
  [voiceover] generating chunk 1/9 (39 words)...
  [voiceover] generating chunk 2/9 (36 words)...
  [voiceover] generating chunk 3/9 (19 words)...
  [voiceover] generating chunk 4/9 (29 words)...
  [voiceover] generating chunk 5/9 (30 words)...
  [voiceover] generating chunk 6/9 (23 words)...
  [voiceover] generating chunk 7/9 (32 words)...
  [voiceover] generating chunk 8/9 (26 words)...
  [voiceover] chunk 8: mlx_audio exited cleanly but produced no audio_*.wav file
  [voiceover] chunk 8 text: 'The Linux 0.11 kernel has been rewritten in idiomatic Rust, allowing it to boot on emulated i386 hardware and run a full init shell coreutils stack.'
  [voiceover] chunk 8: skipping aft

In [27]:
save_checkpoint(call6_1,"till-agent6_1")

[checkpoint] saved 'till-agent6_1' → data/checkpoints/till-agent6_1.json
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s


'data/checkpoints/till-agent6_1.json'

---
## Agent 7: Video Assembly Prompt Generator

Converts Agent 6's approved script into a **per-section shot list** — which story each section belongs to, a stock-footage search query for it, and its timing window within the final audio.

Pivoted from "write a cinematic AI-video prompt" to "extract concrete stock-footage search queries" — real AI video generation costs $0.50-5+/clip on every provider checked (Synthesia, HeyGen, Veo, Grok Imagine), and local generation (Wan2.1 1.3B via mlx-video, tested end-to-end on this project's actual M4 Pro/16GB hardware) needs 10+ minutes and 20GB+ swap per 2-second clip even in its best working configuration. See KNOWN_ISSUES ISSUE-20 for the full evaluation.

| Stage | Model | Job |
|---|---|---|
| Query extraction | `qwen2.5:7b` | 2-5 word stock-footage search query per section — small structured task, doesn't need a heavier model |

**Fallback behavior (pure Python, no LLM cost):** if extraction fails, is malformed, or Ollama's down, falls back to a keyword-sniffed category (security/hardware/software/research/generic) rather than failing the whole run.

**Timing:** word-count-proportional against `audio_duration` — an approximation until real per-section `beat_timestamps` exist from Agent 6.1 (see AGENTS.md dependency note). Includes a defensive check that `word_count` actually matches the sum of section words before trusting any timing math (guards against the same drift documented in ISSUE-18).

In [5]:
from agent_tools.pipeline_cache import load_checkpoint
from agents.agent7 import video_assembly_prompt_node

state = load_checkpoint("till-agent6_1")
call7 = video_assembly_prompt_node(state)

print(f"\n── SHOT LIST ──────────────────────────────────────────")
for shot in call7["shot_list"]:
    marker = "  (fallback)" if shot.get("used_fallback_query") else ""
    title = shot["story_title"] or "(bookend)"
    print(f"{shot['section']:12s} [{shot['start_s']:5.1f}-{shot['end_s']:5.1f}s]  "
          f"rank={shot['story_rank']}  '{shot['query']}'{marker}")
    print(f"             {title}")

fallback_count = sum(1 for s in call7["shot_list"] if s.get("used_fallback_query"))
print(f"\nTotal shots: {len(call7['shot_list'])}")
print(f"Fallback queries used: {fallback_count}/{len(call7['shot_list'])}")

[checkpoint] loaded 'till-agent6_1' (saved: 2026-07-13T22:00:48.623296+00:00)
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s
[agent7] built shot list: 10 sections, 2 used fallback queries
  HOOK         rank=None query='abstract technology particles blue'  (fallback)
  S1_CONTEXT   rank=1 query='NOAA shutdown notice'
  S1_CORE      rank=1 query='former NOAA employees meeting'
  S1_TWIST     rank=1 query='climate data storage'
  S2_HOOK      rank=2 query='clear speech recording'
  S2_CORE      rank=2 query='on-device speech recognition'
  S2_TWIST     rank=2 query='speech recognition process'
  S3_HOOK      rank=3 query='Linux kernel code'
  S3_CORE      rank=3 query='Rust code on screen'
  CTA          rank=None query='abstract technology particles blue'  (fallback)

── SHOT LIST ──────────────────────────────────────────
HOOK         [  0.0-  5.7s]  rank=None  'abstract technology particles blue'  (fallback)
             (bookend)
S1_CONTEXT   [  5.7- 13.0s]  rank=

In [6]:
# ── Save checkpoint after Agent 7 ──────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint
save_checkpoint(call7, 'till-agent7')

[checkpoint] saved 'till-agent7' → data/checkpoints/till-agent7.json
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s


'data/checkpoints/till-agent7.json'

---
## Agent 8: Video Assembler

Consumes Agent 7's shot list + Agent 6.1's stitched audio, renders the final MP4.

**Mode: `reactive`** (default, proven end-to-end against real pipeline audio) — pure PIL/numpy rendered frames: a pulsing/glowing abstract "anchor" orb reacting to the audio's amplitude envelope, a waveform-bar indicator, and source-citation lower-thirds that fade in per story segment using Agent 7's real per-section timing. No ML model, no GPU dependency, no swap/thermal risk — this is the standalone POC that was tested inline in this notebook, now wired to consume real Agent 7 timing instead of a hardcoded single story.

**Mode: `broll`** (scaffolded, NOT tested — needs a real `PEXELS_API_KEY`) — fetches stock footage per shot instead of the reactive graphic. Left unfinished past the fetch step until validated against a live key; calling this mode currently raises `NotImplementedError` on purpose rather than pretending to work.

**Output:** 1080×1920 (YouTube's actual *recommended* Shorts resolution — not the 720×1280 minimum the standalone POC used), saved to `data/video/`.

In [7]:
from agents.agent8 import video_assembler_node
from agent_tools.pipeline_cache import load_checkpoint
import time

print("Rendering + assembling video (reactive mode)...")
print("This renders one PNG frame per video frame (30fps) via PIL, then")
print("muxes against the real audio with ffmpeg — expect several")
print("minutes for a ~80-90s video. No GPU used, so no swap/thermal risk.")
call8=load_checkpoint("till-agent7")

t0 = time.time()
call8o = video_assembler_node(call8, mode="reactive")
elapsed = time.time() - t0

print(f"\n── VIDEO ASSEMBLY RESULT ──────────────────────────────")
print(f"Output path:   {call8o['video_path']}")
print(f"Duration:      {call8o['video_stats']['duration_s']:.1f}s")
print(f"Total frames:  {call8o['video_stats']['total_frames']}")
print(f"Render time:   {call8o['video_stats']['render_time_s']:.1f}s")
print(f"Wall time:     {elapsed:.1f}s (includes ffmpeg encode)")

Rendering + assembling video (reactive mode)...
This renders one PNG frame per video frame (30fps) via PIL, then
muxes against the real audio with ffmpeg — expect several
minutes for a ~80-90s video. No GPU used, so no swap/thermal risk.
[checkpoint] loaded 'till-agent7' (saved: 2026-07-13T22:09:13.161462+00:00)
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s
[agent8] running v6-balanced-2026-07-14 (title=92px orb_r=260 bar_h=460, font-loader macOS-fixed)
[agent8] assembled video: {'total_frames': 2598, 'duration_s': 86.6, 'render_time_s': 64.9, 'output_path': 'data/video/voiceover_20260713_220048.mp4'}

── VIDEO ASSEMBLY RESULT ──────────────────────────────
Output path:   data/video/voiceover_20260713_220048.mp4
Duration:      86.6s
Total frames:  2598
Render time:   64.9s
Wall time:     74.3s (includes ffmpeg encode)


In [8]:
# ── Save checkpoint after Agent 8 ──────────────────────────────────
from agent_tools.pipeline_cache import save_checkpoint
save_checkpoint(call8, 'till-agent8')

[checkpoint] saved 'till-agent8' → data/checkpoints/till-agent8.json
[checkpoint]   stories: 7
[checkpoint]   script:  260 words, ~104s


'data/checkpoints/till-agent8.json'

In [9]:
# Quick sanity checks on today's real output before treating this as
# a usable sample — mirrors the defensive-check habit already used in
# Agent 7's build_shot_list() (word_count validation, missing-story-rank
# errors) rather than assuming the run succeeded just because no
# exception was raised.

import os

video_path = call8o["video_path"]
assert os.path.exists(video_path), f"Expected output video not found: {video_path}"

file_size_mb = os.path.getsize(video_path) / (1024 * 1024)
print(f"File exists: {video_path}")
print(f"File size: {file_size_mb:.1f} MB")

expected_duration = call8o["script"]["audio_duration"]
actual_duration = call8o["video_stats"]["duration_s"]
duration_diff = abs(expected_duration - actual_duration)
print(f"Expected duration: {expected_duration:.1f}s")
print(f"Actual duration:   {actual_duration:.1f}s")
print(f"Difference:        {duration_diff:.2f}s")

if duration_diff > 0.5:
    print("⚠️  Duration mismatch exceeds 0.5s — worth investigating "
          "before treating this as a clean sample (compare against "
          "ISSUE-19's chunk-skip pattern — a dropped audio chunk would "
          "show up here as a duration/frame-count inconsistency).")
else:
    print("✅ Duration matches expected audio length.")

if file_size_mb < 0.5:
    print("⚠️  File size looks suspiciously small — check the file "
          "actually contains video frames, not just a near-empty container.")

File exists: data/video/voiceover_20260713_220048.mp4
File size: 4.7 MB
Expected duration: 86.6s
Actual duration:   86.6s
Difference:        0.00s
✅ Duration matches expected audio length.


---
## Output Organization

Groups this run's finished deliverables — the final video and its source audio — into a single timestamped folder under `output/`, separate from `data/` (which stays as working/intermediate storage: checkpoints, raw audio chunks, frame PNGs).

**Paths:** `data/video/{name}.mp4` + `data/audio/{name}.wav` → copied into `output/{YYYYMMDD_HHMMSS}_{story-slug}/final_video.mp4` + `output/{YYYYMMDD_HHMMSS}_{story-slug}/{name}.wav`. Originals in `data/` are untouched (copied, not moved) — re-running Agent 8 alone doesn't require re-running Agent 6.1.

Kept as its own step, not folded into Agent 8, since "render a video" and "organize finished deliverables" are different concerns — Agent 9/10 (SEO, publishing) will reuse this same organization logic without depending on Agent 8's internals.

In [10]:
from agent_tools.output_organization import organize_run_output, list_runs

run_dir = organize_run_output(call8o)
print(f"\n── OUTPUT ORGANIZED ────────────────────────────────────")
print(f"Run folder: {run_dir}")

print(f"\nAll runs so far:")
for r in list_runs():
    print(f"  {r}")

[output_organization] created output/20260714_034030_climate-gov-was-destroyed-open-data-saved-it-apple-s-new-spe
[output_organization]   video: final_video.mp4 (4.7 MB)
[output_organization]   audio: voiceover_20260713_220048.wav
[output_organization]   metadata: metadata.json

── OUTPUT ORGANIZED ────────────────────────────────────
Run folder: output/20260714_034030_climate-gov-was-destroyed-open-data-saved-it-apple-s-new-spe

All runs so far:
  output/20260714_034030_climate-gov-was-destroyed-open-data-saved-it-apple-s-new-spe
  output/20260714_031435_beavis-ultrasound-pnp-isa-sound-card-replica-cyberpunk-comic
  output/20260714_025814_beavis-ultrasound-pnp-isa-sound-card-replica-cyberpunk-comic


In [11]:
from agent_tools.output_organization import organize_run_output, prune_old_runs

run_dir = organize_run_output(call8o)
prune_old_runs(keep=5)   # deletes anything older than the 5 most recent

[output_organization] created output/20260714_034030_climate-gov-was-destroyed-open-data-saved-it-apple-s-new-spe
[output_organization]   video: final_video.mp4 (4.7 MB)
[output_organization]   audio: voiceover_20260713_220048.wav
[output_organization]   metadata: metadata.json
[output_organization] 3 run(s) found, within the keep=5 limit -- nothing to prune


[]

---
## Showcase Page Generation

Regenerates `docs/audio-showcase.html` and `docs/video-showcase.html` from the most recent runs under `output/` — real story titles, script text, word count, QC iterations, and durations, all pulled from each run's `metadata.json`.

**Manual, not automatic** — run this only when you actually want the public GitHub Pages site updated. A pipeline run finishing doesn't publish anything on its own; this is a deliberate, separate step so a bad run never silently goes live.

Static HTML has no way to read `output/` at view-time, so this bakes the current state into plain HTML ahead of time — see `generate_showcase_pages.py`'s module docstring for the full reasoning.

In [12]:
import subprocess

result = subprocess.run(
    ["python", "agent_tools/generate_showcase_pages.py"],
    cwd=".",
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("── ERROR ──────────────────────────────────────────")
    print(result.stderr)

Scanning output/ for runs...
Found 3 total run(s), taking up to 5 most recent
  skipping 20260714_025814_beavis-ultrasound-pnp-isa-sound-card-replica-cyberpunk-comic: no metadata.json (older run?)

Wrote docs/audio-showcase.html (2 sample(s))
Wrote docs/video-showcase.html (2 sample(s))

